# CoCoNut C 数据集处理

流水线: **dedup/** → 预过滤(条件1) → **filtered/** → SQLite 导入 → 分组处理(条件2-5) → **processed/CoCoNut.jsonl**

## Step 0 — 配置

In [1]:
import json
import sqlite3
import sys
import time
from pathlib import Path

# 将项目根目录加入搜索路径，以便 import utils
PROJECT_ROOT = Path(r"<DATASET_ROOT>\CoCoNut")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import (
    parse_file_path, count_lines_fast, md5_hash,
    compute_group_key, truncate_incomplete_last_line,
    count_matches_in_context, apply_single_replacement,
)

# ── 路径 ─────────────────────────────────────────────────────────────────────
BASE_DIR      = PROJECT_ROOT / "dataset" / "c"
DEDUP_DIR     = BASE_DIR / "dedup"
FILTERED_DIR  = BASE_DIR / "filtered"
PROCESSED_DIR = BASE_DIR / "processed"
DB_PATH       = BASE_DIR / "coconut.db"
JSONL_PATH    = PROCESSED_DIR / "CoCoNut.jsonl"

FILES = ["meta.txt", "add.txt", "rem.txt", "context.txt"]

# ── 过滤阈值 ─────────────────────────────────────────────────────────────────
MAX_CONTEXT_BYTES = 75_000

# ── 创建输出目录 ─────────────────────────────────────────────────────────────
FILTERED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print("tqdm 未安装，使用简单进度输出。")

print(f"去重数据目录  : {DEDUP_DIR}")
print(f"过滤输出目录  : {FILTERED_DIR}")
print(f"SQLite 路径   : {DB_PATH}")
print(f"JSONL 输出    : {JSONL_PATH}")
print(f"字节阈值      : {MAX_CONTEXT_BYTES:,}")

去重数据目录  : <DATASET_ROOT>\CoCoNut\dataset\c\dedup
过滤输出目录  : <DATASET_ROOT>\CoCoNut\dataset\c\filtered
SQLite 路径   : <DATASET_ROOT>\CoCoNut\dataset\c\coconut.db
JSONL 输出    : <DATASET_ROOT>\CoCoNut\dataset\c\processed\CoCoNut.jsonl
字节阈值      : 75,000


## Step 1 — 预过滤（过滤条件 1: context 字节长度 > 阈值）

同时流式读取 dedup/ 下 4 个文件，逐行检查 context 字节长度，
通过的行写入 filtered/。如果 filtered/ 已完成则自动跳过。

In [2]:
def step1_prefilter():
    """执行预过滤，返回 (kept, skipped) 行数。"""

    # ── 恢复检查 ────────────────────────────────────────────────────────────
    all_exist = all((FILTERED_DIR / f).exists() for f in FILES)
    if all_exist:
        counts = [count_lines_fast(FILTERED_DIR / f) for f in FILES]
        if len(set(counts)) == 1 and counts[0] > 0:
            print(f"[跳过] filtered/ 已存在且行数一致 ({counts[0]:,} 行)")
            return counts[0], -1

    # ── 统计 dedup 总行数（用于进度条）──────────────────────────────────────
    total = count_lines_fast(DEDUP_DIR / "meta.txt")
    print(f"dedup/ 总行数: {total:,}")

    kept, skipped = 0, 0
    t0 = time.time()

    src = {f: open(DEDUP_DIR / f, 'r', encoding='utf-8') for f in FILES}
    dst = {f: open(FILTERED_DIR / f, 'w', encoding='utf-8') for f in FILES}

    try:
        pbar = tqdm(total=total, desc="预过滤", unit="行") if HAS_TQDM else None

        for meta, add, rem, ctx in zip(
            src["meta.txt"], src["add.txt"], src["rem.txt"], src["context.txt"]
        ):
            ctx_content = ctx.rstrip('\n')
            if len(ctx_content.encode('utf-8')) > MAX_CONTEXT_BYTES:
                skipped += 1
            else:
                dst["meta.txt"].write(meta)
                dst["add.txt"].write(add)
                dst["rem.txt"].write(rem)
                dst["context.txt"].write(ctx)
                kept += 1

            if pbar:
                pbar.update(1)
            elif (kept + skipped) % 200_000 == 0:
                print(f"  已处理 {kept + skipped:>10,} 行, 保留 {kept:,}, 丢弃 {skipped:,}",
                      flush=True)

        if pbar:
            pbar.close()
    finally:
        for f in src.values(): f.close()
        for f in dst.values(): f.close()

    elapsed = time.time() - t0
    print(f"\n预过滤完成: 保留 {kept:,} 行, 丢弃 {skipped:,} 行, 耗时 {elapsed:.1f}s")
    return kept, skipped

# 保留 1,345,592 行, 丢弃 22,161 行
kept, skipped = step1_prefilter() 

[跳过] filtered/ 已存在且行数一致 (1,345,592 行)


## Step 2 — 导入 SQLite

将 filtered/ 下 4 个文件流式导入 `coconut.db`，建立索引以加速分组查询。
同时创建 `processed_groups` 进度表用于中断恢复。

In [3]:
def step2_import_sqlite():
    """将 filtered/ 数据导入 SQLite，返回导入行数。"""

    conn = sqlite3.connect(str(DB_PATH))
    cur = conn.cursor()

    # ── 建表 ────────────────────────────────────────────────────────────────
    cur.executescript("""
        CREATE TABLE IF NOT EXISTS entries (
            line_no      INTEGER PRIMARY KEY,
            file_path    TEXT    NOT NULL,
            rem          TEXT    NOT NULL,
            add_code     TEXT    NOT NULL,
            context      TEXT    NOT NULL,
            context_hash TEXT    NOT NULL
        );
        CREATE TABLE IF NOT EXISTS processed_groups (
            group_key TEXT PRIMARY KEY,
            status    TEXT NOT NULL
        );
    """)
    conn.commit()

    # ── 恢复检查 ────────────────────────────────────────────────────────────
    existing_rows = cur.execute("SELECT COUNT(*) FROM entries").fetchone()[0]
    filtered_lines = count_lines_fast(FILTERED_DIR / "meta.txt")
    if existing_rows == filtered_lines and existing_rows > 0:
        print(f"[跳过] entries 表已有 {existing_rows:,} 行，与 filtered/ 一致")
        conn.close()
        return existing_rows

    # 如果行数不匹配，清空重新导入
    if existing_rows > 0:
        print(f"entries 表现有 {existing_rows:,} 行，期望 {filtered_lines:,}，清空后重新导入")
        cur.execute("DELETE FROM entries")
        conn.commit()

    # ── 流式导入 ────────────────────────────────────────────────────────────
    print(f"开始导入 {filtered_lines:,} 行到 SQLite ...")
    t0 = time.time()

    src = {
        "meta": open(FILTERED_DIR / "meta.txt",    'r', encoding='utf-8'),
        "add":  open(FILTERED_DIR / "add.txt",     'r', encoding='utf-8'),
        "rem":  open(FILTERED_DIR / "rem.txt",     'r', encoding='utf-8'),
        "ctx":  open(FILTERED_DIR / "context.txt", 'r', encoding='utf-8'),
    }

    BATCH = 50_000
    batch_buf = []
    inserted = 0

    pbar = tqdm(total=filtered_lines, desc="SQLite 导入", unit="行") if HAS_TQDM else None

    try:
        for lineno, (meta, add, rem, ctx) in enumerate(
            zip(src["meta"], src["add"], src["rem"], src["ctx"]),
            start=1
        ):
            fp = parse_file_path(meta)
            ctx_stripped = ctx.rstrip('\n')
            c_hash = md5_hash(ctx_stripped)

            batch_buf.append((
                lineno,
                fp,
                rem.rstrip('\n'),
                add.rstrip('\n'),
                ctx_stripped,
                c_hash,
            ))

            if len(batch_buf) >= BATCH:
                cur.executemany(
                    "INSERT INTO entries VALUES (?,?,?,?,?,?)",
                    batch_buf,
                )
                conn.commit()
                inserted += len(batch_buf)
                if pbar:
                    pbar.update(len(batch_buf))
                else:
                    print(f"  {inserted:>10,} / {filtered_lines:,}", flush=True)
                batch_buf.clear()

        # 剩余数据
        if batch_buf:
            cur.executemany(
                "INSERT INTO entries VALUES (?,?,?,?,?,?)",
                batch_buf,
            )
            conn.commit()
            inserted += len(batch_buf)
            if pbar:
                pbar.update(len(batch_buf))

        if pbar:
            pbar.close()
    finally:
        for f in src.values(): f.close()

    # ── 建索引 ──────────────────────────────────────────────────────────────
    print("创建索引 ...")
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_group
        ON entries(file_path, context_hash)
    """)
    conn.commit()
    conn.close()

    elapsed = time.time() - t0
    print(f"导入完成: {inserted:,} 行, 耗时 {elapsed:.1f}s")
    return inserted


imported = step2_import_sqlite()

[跳过] entries 表已有 1,345,592 行，与 filtered/ 一致


## Step 3 — 分组处理 + 过滤条件 2-5 + JSONL 输出

按 `(file_path, context_hash)` 分组，对每组：
1. 过滤条件 3: 任意 rem 为空 → 跳过
2. 逐个应用 (rem, add) 到 context:
   - 过滤条件 4: rem 匹配 > 1 次 → 跳过
   - 过滤条件 5: rem 匹配 0 次 → 跳过
3. 过滤条件 2: fixed_code 字节 > 阈值 → 跳过
4. 通过所有条件 → 写入 JSONL，标记完成

内置中断恢复：利用 `processed_groups` 表跳过已完成的组，
启动时截断 JSONL 残行。

In [4]:
def step3_process_groups():
    """分组处理并输出 JSONL。"""

    # ── JSONL 残行截断（中断恢复）─────────────────────────────────────────
    valid_lines = truncate_incomplete_last_line(JSONL_PATH)
    print(f"JSONL 现有有效行数: {valid_lines:,}")

    conn = sqlite3.connect(str(DB_PATH))
    cur = conn.cursor()

    # ── 查询所有分组 ────────────────────────────────────────────────────────
    cur.execute("SELECT DISTINCT file_path, context_hash FROM entries")
    all_groups = cur.fetchall()
    total_groups = len(all_groups)
    print(f"总分组数: {total_groups:,}")

    # ── 已处理的分组 ────────────────────────────────────────────────────────
    cur.execute("SELECT group_key FROM processed_groups")
    done_keys = set(row[0] for row in cur.fetchall())
    print(f"已处理分组: {len(done_keys):,}, 待处理: {total_groups - len(done_keys):,}")

    # ── 统计计数器 ──────────────────────────────────────────────────────────
    stats = {
        "completed": 0, "skipped_filter2": 0, "skipped_filter3": 0,
        "skipped_filter4": 0, "skipped_filter5": 0, "skipped_already": 0,
    }

    t0 = time.time()
    pbar = tqdm(total=total_groups, desc="分组处理", unit="组") if HAS_TQDM else None

    jsonl_file = open(JSONL_PATH, 'a', encoding='utf-8')

    try:
        for idx, (fp, c_hash) in enumerate(all_groups):
            gk = compute_group_key(fp, c_hash)

            # ── 跳过已完成 ──────────────────────────────────────────────────
            if gk in done_keys:
                stats["skipped_already"] += 1
                if pbar:
                    pbar.update(1)
                continue

            # ── 加载该组所有条目（按行号排序）─────────────────────────────────
            cur.execute(
                "SELECT rem, add_code, context FROM entries "
                "WHERE file_path = ? AND context_hash = ? "
                "ORDER BY line_no",
                (fp, c_hash),
            )
            rows = cur.fetchall()
            context_original = rows[0][2]
            changes = [(r[0], r[1]) for r in rows]

            # ── 过滤条件 3: 任意 rem 为空 ───────────────────────────────────
            if any(rem.strip() == '' for rem, _ in changes):
                _mark_group(cur, conn, gk, "skipped_filter3", done_keys)
                stats["skipped_filter3"] += 1
                if pbar: pbar.update(1)
                continue

            # ── 逐个应用变更 ────────────────────────────────────────────────
            current_ctx = context_original
            skip_reason = None

            for rem, add in changes:
                match_count, strategy = count_matches_in_context(current_ctx, rem)

                if match_count == 0:
                    skip_reason = "skipped_filter5"
                    break
                if match_count > 1:
                    skip_reason = "skipped_filter4"
                    break

                new_ctx, n, _ = apply_single_replacement(current_ctx, rem, add)
                if n != 1:
                    skip_reason = "skipped_filter5"
                    break
                current_ctx = new_ctx

            if skip_reason:
                _mark_group(cur, conn, gk, skip_reason, done_keys)
                stats[skip_reason] += 1
                if pbar: pbar.update(1)
                continue

            # ── 过滤条件 2: fixed_code 字节长度 > 阈值 ─────────────────────
            if len(current_ctx.encode('utf-8')) > MAX_CONTEXT_BYTES:
                _mark_group(cur, conn, gk, "skipped_filter2", done_keys)
                stats["skipped_filter2"] += 1
                if pbar: pbar.update(1)
                continue

            # ── 写入 JSONL ──────────────────────────────────────────────────
            entry = {
                "id": fp,
                "language": "C",
                "buggy_code": context_original,
                "fixed_code": current_ctx,
            }
            jsonl_file.write(json.dumps(entry, ensure_ascii=False) + '\n')
            jsonl_file.flush()

            _mark_group(cur, conn, gk, "completed", done_keys)
            stats["completed"] += 1

            if pbar:
                pbar.update(1)
            elif (idx + 1) % 10_000 == 0:
                elapsed = time.time() - t0
                print(f"  {idx + 1:>10,} / {total_groups:,}  "
                      f"完成 {stats['completed']:,}  耗时 {elapsed:.0f}s",
                      flush=True)

    finally:
        jsonl_file.close()
        conn.close()
        if pbar:
            pbar.close()

    elapsed = time.time() - t0
    print(f"\n处理完成，耗时 {elapsed / 60:.1f} 分钟")
    print("─" * 50)
    for k, v in stats.items():
        print(f"  {k:<20s}: {v:>10,}")
    print("─" * 50)
    total_output = stats['completed']
    print(f"  JSONL 输出条目总数  : {total_output:,}")


def _mark_group(cur, conn, gk, status, done_keys):
    """在 processed_groups 中记录分组处理状态。"""
    cur.execute(
        "INSERT OR REPLACE INTO processed_groups VALUES (?, ?)",
        (gk, status),
    )
    conn.commit()
    done_keys.add(gk)


step3_process_groups()

JSONL 现有有效行数: 144,274
总分组数: 555,043
已处理分组: 555,043, 待处理: 0


分组处理:   0%|          | 0/555043 [00:00<?, ?组/s]


处理完成，耗时 0.0 分钟
──────────────────────────────────────────────────
  completed           :          0
  skipped_filter2     :          0
  skipped_filter3     :          0
  skipped_filter4     :          0
  skipped_filter5     :          0
  skipped_already     :    555,043
──────────────────────────────────────────────────
  JSONL 输出条目总数  : 0


## Step 4 — 内容去重

不同 `id`（文件路径）可能对应完全相同的 `buggy_code` + `fixed_code`（同一源文件被复制到项目的不同路径下）。
按 `MD5(buggy_code || fixed_code)` 去重，保留每组首次出现的记录。

In [5]:
def step4_content_dedup():
    """按 (buggy_code, fixed_code) 内容指纹去重 JSONL。"""

    if not JSONL_PATH.exists():
        print("CoCoNut.jsonl 不存在，跳过去重。")
        return

    before_count = count_lines_fast(JSONL_PATH)
    print(f"去重前记录数: {before_count:,}")

    tmp_path = JSONL_PATH.with_suffix('.jsonl.tmp')
    seen = set()
    kept, dropped = 0, 0

    with open(JSONL_PATH, 'r', encoding='utf-8') as fin, \
         open(tmp_path, 'w', encoding='utf-8') as fout:
        for line in fin:
            obj = json.loads(line)
            fingerprint = md5_hash(obj['buggy_code'] + '||' + obj['fixed_code'])
            if fingerprint in seen:
                dropped += 1
                continue
            seen.add(fingerprint)
            fout.write(line)
            kept += 1

    # 用去重后的文件替换原文件
    tmp_path.replace(JSONL_PATH)

    print(f"去重后记录数: {kept:,}")
    print(f"移除重复记录: {dropped:,}")
    size_mb = JSONL_PATH.stat().st_size / 1e6
    print(f"文件大小    : {size_mb:.1f} MB")


step4_content_dedup()

去重前记录数: 144,274
去重后记录数: 144,274
移除重复记录: 0
文件大小    : 915.4 MB


## Step 5 — 长度过滤

过滤过长或过短的代码片段。
- MAX_CODE_LEN = 65535 (默认)
- MIN_CODE_LEN = 35 (默认)

如果 buggy_code 或 fixed_code 的长度超过上限或少于下限，则舍弃该记录。

In [6]:
def step5_length_filter():
    """按代码长度过滤 JSONL，移除过长或过短的记录。"""

    MAX_CODE_LEN = 65_535  # 默认最大长度
    MIN_CODE_LEN = 35      # 默认最小长度

    if not JSONL_PATH.exists():
        print("CoCoNut.jsonl 不存在，跳过长度过滤。")
        return

    before_count = count_lines_fast(JSONL_PATH)
    print(f"过滤前记录数: {before_count:,}")
    print(f"长度限制: [{MIN_CODE_LEN}, {MAX_CODE_LEN}]")

    tmp_path = JSONL_PATH.with_suffix('.jsonl.len.tmp')
    kept, dropped_too_long, dropped_too_short = 0, 0, 0

    with open(JSONL_PATH, 'r', encoding='utf-8') as fin, \
         open(tmp_path, 'w', encoding='utf-8') as fout:
        for line in fin:
            obj = json.loads(line)
            buggy_len = len(obj['buggy_code'])
            fixed_len = len(obj['fixed_code'])

            # 检查是否过长
            if buggy_len > MAX_CODE_LEN or fixed_len > MAX_CODE_LEN:
                dropped_too_long += 1
                continue

            # 检查是否过短
            if buggy_len < MIN_CODE_LEN or fixed_len < MIN_CODE_LEN:
                dropped_too_short += 1
                continue

            fout.write(line)
            kept += 1

    # 用过滤后的文件替换原文件
    tmp_path.replace(JSONL_PATH)

    print(f"过滤后记录数: {kept:,}")
    print(f"移除过长记录: {dropped_too_long:,}")
    print(f"移除过短记录: {dropped_too_short:,}")
    print(f"总移除记录: {dropped_too_long + dropped_too_short:,}")
    size_mb = JSONL_PATH.stat().st_size / 1e6
    print(f"文件大小    : {size_mb:.1f} MB")


step5_length_filter()

过滤前记录数: 144,274
长度限制: [35, 65535]
过滤后记录数: 144,274
移除过长记录: 0
移除过短记录: 0
总移除记录: 0
文件大小    : 915.4 MB


## 验证

In [7]:
if JSONL_PATH.exists():
    n = count_lines_fast(JSONL_PATH)
    size_mb = JSONL_PATH.stat().st_size / 1e6
    print(f"CoCoNut.jsonl: {n:,} 条记录, {size_mb:.1f} MB") # 244,191 条记录

    # 展示前 3 条
    with open(JSONL_PATH, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            obj = json.loads(line)
            print(f"\n── 记录 {i+1} ──")
            print(f"  id          : {obj['id'][:80]}...")
            print(f"  language    : {obj['language']}")
            print(f"  buggy_code  : {len(obj['buggy_code']):,} 字符")
            print(f"  fixed_code  : {len(obj['fixed_code']):,} 字符")
else:
    print("CoCoNut.jsonl 尚未生成。")

CoCoNut.jsonl: 144,274 条记录, 915.4 MB

── 记录 1 ──
  id          : /local/tlutelli/issta_data/temp/c/2005_temp/2005/10355/01b6ca963c9be8447e7e0b9c5...
  language    : C
  buggy_code  : 1,104 字符
  fixed_code  : 1,261 字符

── 记录 2 ──
  id          : /local/tlutelli/issta_data/temp/c/2005_temp/2005/10355/0286b9f6145f9664f6b0e196e...
  language    : C
  buggy_code  : 293 字符
  fixed_code  : 293 字符

── 记录 3 ──
  id          : /local/tlutelli/issta_data/temp/c/2005_temp/2005/10355/0286b9f6145f9664f6b0e196e...
  language    : C
  buggy_code  : 716 字符
  fixed_code  : 619 字符
